In [ ]:
# Projeto de Tratamento de Dados - Acidentes nas Rodovias Federais (PRF)

# Integrantes:
# Nome: Rafael Augusto Balko de Souza - RA: 2411551016
# Nome: Rafael Sanabria Zibordi - RA: 

In [ ]:
import pandas as pd
import folium
from folium.plugins import MiniMap, Fullscreen, Geocoder, MousePosition

print("Pandas versão:", pd.__version__)
print("Folium versão:", folium.__version__)

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [ ]:
# Carregando o dataset de acidentes de 2025
df_original = pd.read_csv('datatran2025.csv', sep=';', encoding='UTF-8')

# Exibindo o tamanho da base e as duas primeiras linhas
print(df_original.shape[0], 'linhas')
print(df_original.shape[1], 'colunas')
df_original.head(5)

In [ ]:
df_tratado = df_original.copy()

df_tratado['data_inversa'] = pd.to_datetime(df_tratado['data_inversa'], format='%d/%m/%Y', errors='coerce')
df_tratado[['data_inversa']].dtypes

In [ ]:
# Gerando o ranking dos estados com mais acidentes
df_uf = df_tratado.value_counts(subset=['uf']) \
                  .to_frame().reset_index() \
                  .rename(columns={'count': 'total_acidentes'})

# Salvando a tabela gerada em um arquivo CSV independente
df_uf.to_csv('ocorrencias_por_estado_prf.csv', index=False)

# Exibindo o Top 10 na tela
df_uf.head(10)

In [ ]:
# Gerando o ranking das causas de acidentes
df_causas = df_tratado.value_counts(subset=['causa_acidente']) \
                      .to_frame().reset_index() \
                      .rename(columns={'count': 'total_ocorrencias'})

df_causas.to_csv('causas_acidentes_prf.csv', index=False)
df_causas.head(10)

In [ ]:
import folium
from folium.plugins import MarkerCluster, MiniMap, Fullscreen, Geocoder, MousePosition

df_fatais = df_tratado.query('mortos > 0').copy()
df_fatais['data_inversa'] = pd.to_datetime(df_fatais['data_inversa'], format='%d/%m/%Y', errors='coerce')
df_fatais = df_fatais.dropna(subset=['data_inversa', 'latitude', 'longitude'])

df_fatais['latitude'] = df_fatais['latitude'].str.replace(',', '.').astype(float)
df_fatais['longitude'] = df_fatais['longitude'].str.replace(',', '.').astype(float)

print(f"Foram registrados {df_fatais.shape[0]} acidentes fatais em rodovias federais em 2025.")

# criando e separando o top 5 das principais causas dos acidentes
top_causas = df_fatais['causa_acidente'].value_counts().head(5).index.tolist()

# criando o mapa base centralizado no Brasil
mapa_prf = folium.Map(location=[-14.2350, -51.9253], zoom_start=4, tiles='OpenStreetMap')

# criando as ferramentas para navegação no mapa
MiniMap().add_to(mapa_prf)
Fullscreen().add_to(mapa_prf)
Geocoder().add_to(mapa_prf)
MousePosition().add_to(mapa_prf)

# criando os filtros no mapa e os clusters
# criamos um dicionário para guardar um cluster dedicado a cada causa do Top 5 + Outras Causas
camadas_causas = {}

for causa in top_causas:
    # Cria a camada que aparece na legenda
    grupo = folium.FeatureGroup(name=f"{causa}").add_to(mapa_prf)
    # Cria o cluster de bolinhas dentro dessa camada específica
    camadas_causas[causa] = MarkerCluster(showCoverageOnHover=False).add_to(grupo)

# Criando uma camada extra para as causas menos frequentes
grupo_outras = folium.FeatureGroup(name="Causa: Outras Causas").add_to(mapa_prf)
camadas_causas['Outras'] = MarkerCluster(showCoverageOnHover=False).add_to(grupo_outras)

# Iterando sobre as linhas para distribuir os pontos em seus respectivos filtros
for index, linha in df_fatais.iterrows():
    
    br_info = int(linha['br']) if pd.notna(linha['br']) else "N/A"
    causa_atual = linha['causa_acidente']
    
    texto_popup = f"""
    <b>Data:</b> {linha['data_inversa'].strftime('%d/%m/%Y')}<br>
    <b>Local:</b> {linha['uf']} - BR {br_info} | KM {linha['km']}<br>
    <b>Município:</b> {linha['municipio']}<br>
    <b>Causa:</b> {causa_atual}<br>
    <b>Mortes:</b> <font color='red'><b>{linha['mortos']}</b></font>
    """
    
    # Define em qual caixinha/cluster essa linha de dados vai entrar
    cluster_destino = camadas_causas[causa_atual] if causa_atual in top_causas else camadas_causas['Outras']
    
    folium.CircleMarker(
        location=[linha['latitude'], linha['longitude']], 
        radius=5,
        popup=folium.Popup(texto_popup, max_width=300),
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7
    ).add_to(cluster_destino)

# habilitando o painel lateral com as caixinhas de seleção das causas
folium.LayerControl(collapsed=False).add_to(mapa_prf)

mapa_prf.save('mapa_acidentes_fatais.html')
mapa_prf